# 01 ComfyUI：啟動、健康檢查、等待 Agent 的 job

**執行階段 > 全部執行**：啟動 ComfyUI → API 測試（不需要模型）→ 開始等待 `jobs/pending` 裡的 job。

AI Agent 模式就是讓這本 notebook 開著：Agent 用工具建立 job，這裡負責執行並把結果寫回 Google Drive。

GPU 要自己選（執行階段 > 變更執行階段類型）：Z-Image 需要 **L4** 以上，MiniMax H3 需要 **A100**。GPU 不夠的 job 會留在佇列裡等，不會被標成失敗。

In [ ]:
#@title 1. 掛載 Google Drive，載入 Core { display-mode: "form" }
ROOT = "/content/drive/MyDrive/AI-Workflow"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import sys
sys.path.insert(0, ROOT)
from controller import notebook as nb
session = nb.connect(ROOT, mount=False)

In [ ]:
#@title 2. 啟動 ComfyUI 與健康檢查 { display-mode: "form" }
stats = nb.start_comfyui(session)

In [ ]:
#@title 3. ComfyUI API 測試 { display-mode: "form" }
test = nb.api_test(session)

In [ ]:
#@title 4. 等待並執行 job（Agent 模式） { display-mode: "form" }
#@markdown 取消勾選就只做上面的啟動與測試，不進入等待。
wait_for_jobs = True  #@param {type:"boolean"}
#@markdown 佇列空了這麼多分鐘後停止等待：
idle_minutes = 10  #@param {type:"integer"}
#@markdown 停止後釋放 runtime（不再消耗運算單元）。取消勾選的話 runtime 會繼續計費，要自己中斷。
release_runtime = True  #@param {type:"boolean"}
if wait_for_jobs:
    summary = nb.serve(session, idle_minutes, release_runtime)